# 第11章 指示チューニング

## 11.2 指示チューニングの実装

### 11.2.1 環境の準備

In [1]:
!pip install datasets transformers[torch,sentencepiece] trltrl==0.8.6 peft bitsandbytes

ERROR: Could not find a version that satisfies the requirement trltrl==0.8.6 (from versions: none)
ERROR: No matching distribution found for trltrl==0.8.6


In [2]:
from transformers.trainer_utils import set_seed
set_seed(42)

### 11.2.2 データセットの準備

In [3]:
from pprint import pprint
from datasets import load_dataset

dataset = load_dataset("llm-book/oasst1-21k-ja", split="train")
pprint(dataset)

Dataset({
    features: ['conversation'],
    num_rows: 21164
})


In [4]:
from pprint import pprint

# pprintで見やすく表示する
pprint(dataset[0])

{'conversation': [{'content': 'こんにちは！', 'role': 'user'},
                  {'content': 'こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。',
                   'role': 'assistant'},
                  {'content': '世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。',
                   'role': 'user'},
                  {'content': '世界中の国をアルファベット順に並べたリストと、その国の推定人口です：\n'
                              '\n'
                              'アフガニスタン: 38,928,346 アルバニア: 2,877,797 '
                              'アルジェリア：44,344,744 アンドラ: 77,265 アンゴラ: 32,878,272 '
                              'アンティグア・バーブーダ: 97,929 アルゼンチン: 45,195,774 アルメニア: '
                              '2,977,600 オーストラリア: 25,499,884 オーストリア: 9,006,398 '
                              'アゼルバイジャン: 10,134,604 バハマ：393,248 バーレーン: '
                              '1,714,571 バングラデシュ: 164,689,383164,689,383 '
                              'バルバドス: 287,375 ベラルーシ: 9,449,323 ベルギー: '
                              '11,589,623 ベリーズ: 397,628 ベナン: 12,123,200 ブータン: 

### 11.2.3 チャットテンプレートの作成

In [5]:
from transformers import AutoTokenizer

base_model_name = "tokyotech-llm/Swallow-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

In [6]:
tokenizer.chat_template = """\
{%- for message in messages %}
{%- if message['role'] == 'user' %}
{{ bos_token + 'ユーザ：' + message['content'] + eos_token }}
{%- elif message['role'] == 'assistant' %}
{{ bos_token + 'アシスタント：'  + message['content'] + eos_token }}
{%- endif %}
{% if loop.last and add_generation_prompt %}
{{ bos_token + 'アシスタント：' }}
{%- endif %}
{% endfor %}\
"""

In [7]:
# デフォルトではトークナイズかつID化されたトークンのリストが返されるが、
# ここではtokenize=Falseとしてトークナイズ前の文字列を返すように設定
chat_text = tokenizer.apply_chat_template(
    dataset[0]["conversation"], tokenize=False
)
# 発話間に改行が含まれないため、見やすくするためにeos_tokenを改行に置換
print(chat_text.replace(tokenizer.eos_token, "\n"))

<s>ユーザ：こんにちは！
<s>アシスタント：こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。
<s>ユーザ：世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。
<s>アシスタント：世界中の国をアルファベット順に並べたリストと、その国の推定人口です：

アフガニスタン: 38,928,346 アルバニア: 2,877,797 アルジェリア：44,344,744 アンドラ: 77,265 アンゴラ: 32,878,272 アンティグア・バーブーダ: 97,929 アルゼンチン: 45,195,774 アルメニア: 2,977,600 オーストラリア: 25,499,884 オーストリア: 9,006,398 アゼルバイジャン: 10,134,604 バハマ：393,248 バーレーン: 1,714,571 バングラデシュ: 164,689,383164,689,383 バルバドス: 287,375 ベラルーシ: 9,449,323 ベルギー: 11,589,623 ベリーズ: 397,628 ベナン: 12,123,200 ブータン: 754,288 ボリビア: 11,673,095 ボスニア・ヘルツェゴビナ: 3,279,000 ボツワナ：2,335,814 ブラジル: 212,559,417 ブルネイ: 441,484 ブルガリア: 7,034,879 ブルキナファソ: 20,903,273 ブルンジ: 11,890,781 カボベルデ: 555,987 カンボジア: 16,205,218 カメルーン: 26,545,863 カナダ：37,742,154 中央アフリカ共和国4,829,767 チャド: 16,425,864 チリ: 19,116,201 中国: 1,439,323,776 コロンビア: 50,882,891 コモロ: 869,601 コンゴ民主共和国：87,534,403 コンゴ共和国5,457,821 コスタリカ5,094,118 コートジボワール: 26,378,274 クロアチア: 4,105,267 キューバ: 11,239,224 キプロス：1,207,359 チェコ：10,708,919 デンマーク：5,792,2025,792,202



In [8]:
# 会話データの末尾のアシスタントの発話を除き、生成を促すための文字列を追加
chat_text = tokenizer.apply_chat_template(
    dataset[0]["conversation"][:-1],
    tokenize=False,
    add_generation_prompt=True,
)
print(chat_text.replace(tokenizer.eos_token, "\n"))

<s>ユーザ：こんにちは！
<s>アシスタント：こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。
<s>ユーザ：世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。
<s>アシスタント：


### 11.2.4 トークンIDへの変換

In [9]:
# チャットテンプレートを適用してトークンIDに変換
tokenized_dataset = [
    tokenizer.apply_chat_template(item["conversation"])
    for item in dataset
]

In [10]:
# トークンIDに変換されたデータセットの先頭を表示
token_ids = tokenized_dataset[0]
print("トークンID:", token_ids)
print("トークン:", tokenizer.convert_ids_to_tokens(token_ids["input_ids"]))

トークンID: {'input_ids': [1, 39944, 30383, 33328, 30584, 2, 1, 36166, 32113, 32040, 30383, 33328, 30584, 31622, 32425, 32099, 31111, 30697, 36451, 30199, 32002, 30458, 32009, 32006, 32075, 30330, 31502, 39682, 31622, 32277, 32059, 30267, 31502, 30458, 32090, 30412, 32903, 30466, 32059, 30267, 2, 1, 39944, 30383, 32198, 30199, 32668, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 30330, 32728, 30199, 30356, 30199, 35620, 30396, 32903, 30466, 32059, 30267, 2, 1, 36166, 32113, 32040, 30383, 32198, 30275, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 37231, 30366, 33634, 30364, 30330, 32016, 30356, 30199, 40875, 35620, 32001, 30383, 0, 34874, 41714, 39676, 29901, 0, 29941, 29947, 29892, 29929, 29906, 29947, 29892, 29941, 29946, 29953, 0, 33997, 32873, 29901, 0, 29906, 29892, 29947, 29955, 29955, 29892, 29955, 29929, 29955, 0, 32513, 32882, 32166, 30383, 29946, 29946, 29892, 29941, 29946, 29946, 29892, 29955, 29946, 29946, 0, 32709, 32498, 29901, 0, 29955, 29955, 29892, 2990

In [11]:
tokenizer.pad_token = tokenizer.unk_token

In [18]:
from trl import DataCollatorForCompletionOnlyLM

bos = tokenizer.bos_token
collator = DataCollatorForCompletionOnlyLM(
    instruction_template=bos + "ユーザ:",      # ユーザの発話開始を示す文字列
    response_template=bos + "アシスタント：", # アシスタントの返答開始を示す文字列
    tokenizer=tokenizer,  # トークナイザ
)
# トークナイズされたデータセットの先頭をミニバッチ構築処理
batch = collator(tokenized_dataset[:1])
input_ids = batch["input_ids"][0]
labels = batch["labels"][0]

/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py:183: UserWarning: Could not find instruction key `<s>ユーザ:` in the following instance: <s> ユーザ ： こんにちは ！ </s> <s> アシ スタ ント ： こんにちは ！ ご 質 問 や お 困り の こと が あり まし たら 、 何 でも ご 相談 ください 。 何 が 必要 か 教え て ください 。 </s> <s> ユーザ ： 世界 の すべて の 国 を アル ファ ベット 順 に 、 それぞれ の 国 の 人口 を 教え て ください 。 </s> <s> アシ スタ ント ： 世界 中 の 国 を アル ファ ベット 順 に 並べ た リスト と 、 その 国 の 推定 人口 です ： <unk> アフ ガニ スタン : <unk> 3 8 , 9 2 8 , 3 4 6 <unk> アルバ ニア : <unk> 2 , 8 7 7 , 7 9 7 <unk> アル ジェ リア ： 4 4 , 3 4 4 , 7 4 4 <unk> アン ドラ : <unk> 7 7 , 2 6 5 <unk> アン ゴ ラ : <unk> 3 2 , 8 7 8 , 2 7 2 <unk> アン ティ グ ア ・ バー ブー ダ : <unk> 9 7 , 9 2 9 <unk> アル ゼン チン : <unk> 4 5 , 1 9 5 , 7 7 4 <unk> アル メ ニア : <unk> 2 , 9 7 7 , 6 0 0 <unk> オーストラリア : <unk> 2 5 , 4 9 9 , 8 8 4 <unk> オー スト リア : <unk> 9 , 0 0 6 , 3 9 8 <unk> ア ゼル バイ ジャン : <unk> 1 0 , 1 3 4 , 6 0 4 <unk> バ ハマ ： 3 9 3 , 2 4 8 <unk> バー レー ン : <unk> 1 , 7 1 4 , 5 7 1 <unk> バン グラ デ シュ : <unk> 1 6 4 , 6 8 9 , 3 8 3 1 6 4 , 6 8 9 , 3 8 3 <

In [22]:
pprint(batch)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         

In [28]:
print(tokenized_dataset[:1])
print(tokenized_dataset[:1][0].keys())

[{'input_ids': [1, 39944, 30383, 33328, 30584, 2, 1, 36166, 32113, 32040, 30383, 33328, 30584, 31622, 32425, 32099, 31111, 30697, 36451, 30199, 32002, 30458, 32009, 32006, 32075, 30330, 31502, 39682, 31622, 32277, 32059, 30267, 31502, 30458, 32090, 30412, 32903, 30466, 32059, 30267, 2, 1, 39944, 30383, 32198, 30199, 32668, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 30330, 32728, 30199, 30356, 30199, 35620, 30396, 32903, 30466, 32059, 30267, 2, 1, 36166, 32113, 32040, 30383, 32198, 30275, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 37231, 30366, 33634, 30364, 30330, 32016, 30356, 30199, 40875, 35620, 32001, 30383, 0, 34874, 41714, 39676, 29901, 0, 29941, 29947, 29892, 29929, 29906, 29947, 29892, 29941, 29946, 29953, 0, 33997, 32873, 29901, 0, 29906, 29892, 29947, 29955, 29955, 29892, 29955, 29929, 29955, 0, 32513, 32882, 32166, 30383, 29946, 29946, 29892, 29941, 29946, 29946, 29892, 29955, 29946, 29946, 0, 32709, 32498, 29901, 0, 29955, 29955, 29892, 29906, 2995

In [19]:
print("入力トークンID:", input_ids)
print("正解ラベル:", labels)

入力トークンID: tensor([    1, 39944, 30383, 33328, 30584,     2,     1, 36166, 32113, 32040,
        30383, 33328, 30584, 31622, 32425, 32099, 31111, 30697, 36451, 30199,
        32002, 30458, 32009, 32006, 32075, 30330, 31502, 39682, 31622, 32277,
        32059, 30267, 31502, 30458, 32090, 30412, 32903, 30466, 32059, 30267,
            2,     1, 39944, 30383, 32198, 30199, 32668, 30199, 30356, 30396,
        32513, 32148, 39383, 33045, 30353, 30330, 32728, 30199, 30356, 30199,
        35620, 30396, 32903, 30466, 32059, 30267,     2,     1, 36166, 32113,
        32040, 30383, 32198, 30275, 30199, 30356, 30396, 32513, 32148, 39383,
        33045, 30353, 37231, 30366, 33634, 30364, 30330, 32016, 30356, 30199,
        40875, 35620, 32001, 30383,     0, 34874, 41714, 39676, 29901,     0,
        29941, 29947, 29892, 29929, 29906, 29947, 29892, 29941, 29946, 29953,
            0, 33997, 32873, 29901,     0, 29906, 29892, 29947, 29955, 29955,
        29892, 29955, 29929, 29955,     0, 32513, 3288

In [20]:
import itertools

segments_to_fit: list[list[int]] = []
segments_to_ignore: list[list[int]] = []
# ラベルが-100である箇所とそうでない箇所ごとにグルーピング
for key, group in itertools.groupby(
    range(len(input_ids)), key=lambda i: labels[i] == -100
):
    group = list(group)
    if key:
        segments_to_ignore.append(group)
    else:
        segments_to_fit.append(group)

In [21]:
print("---- 損失を計算しない部分 ----")
for seg in segments_to_ignore:
    print(tokenizer.decode(input_ids[seg]))
    print()

print("---- 損失を計算する部分 ----")
for seg in segments_to_fit:
    print(tokenizer.decode(input_ids[seg]))
    print()

---- 損失を計算しない部分 ----
<s> ユーザ ： こんにちは ！ </s> <s> アシ スタ ント ： こんにちは ！ ご 質 問 や お 困り の こと が あり まし たら 、 何 でも ご 相談 ください 。 何 が 必要 か 教え て ください 。 </s> <s> ユーザ ： 世界 の すべて の 国 を アル ファ ベット 順 に 、 それぞれ の 国 の 人口 を 教え て ください 。 </s> <s> アシ スタ ント ： 世界 中 の 国 を アル ファ ベット 順 に 並べ た リスト と 、 その 国 の 推定 人口 です ： <unk> アフ ガニ スタン : <unk> 3 8 , 9 2 8 , 3 4 6 <unk> アルバ ニア : <unk> 2 , 8 7 7 , 7 9 7 <unk> アル ジェ リア ： 4 4 , 3 4 4 , 7 4 4 <unk> アン ドラ : <unk> 7 7 , 2 6 5 <unk> アン ゴ ラ : <unk> 3 2 , 8 7 8 , 2 7 2 <unk> アン ティ グ ア ・ バー ブー ダ : <unk> 9 7 , 9 2 9 <unk> アル ゼン チン : <unk> 4 5 , 1 9 5 , 7 7 4 <unk> アル メ ニア : <unk> 2 , 9 7 7 , 6 0 0 <unk> オーストラリア : <unk> 2 5 , 4 9 9 , 8 8 4 <unk> オー スト リア : <unk> 9 , 0 0 6 , 3 9 8 <unk> ア ゼル バイ ジャン : <unk> 1 0 , 1 3 4 , 6 0 4 <unk> バ ハマ ： 3 9 3 , 2 4 8 <unk> バー レー ン : <unk> 1 , 7 1 4 , 5 7 1 <unk> バン グラ デ シュ : <unk> 1 6 4 , 6 8 9 , 3 8 3 1 6 4 , 6 8 9 , 3 8 3 <unk> バル バ ド ス : <unk> 2 8 7 , 3 7 5 <unk> ベ ラ ルー シ : <unk> 9 , 4 4 9 , 3 2 3 <unk> ベル ギー : <unk> 1 1 , 5 8 9 , 6 2 3 <unk> ベ リ